# Building Retry and Fallback Logic for Production LLM Applications


> Model deployment and credential setup for this demo are covered in the accompanying `demo-01-Building Retry and Fallback Logic for Production LLM Applications.sh`. Both `gpt-5-mini` and `gpt-4.1-mini` should already be deployed and Live in Foundry before running this notebook.

> 1. Install resilience libraries
> 2. Imports, client setup, and model names
> 3. Prove the real deployment works
> 4. Break it: raw crash, no handling
> 5. Add retry only
> 6. Add fallback on top
> 7. Put it together: one reusable function

## Install resilience libraries


In [ ]:
pip install tenacity openai


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import libraries

In [ ]:
import time, os

from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError

from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type

## Cell 3 — Client setup



In [ ]:
client = OpenAI(
    base_url="https://loony-foundry.openai.azure.com/openai/v1",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

client

## Model names + timeout values

Two timeout values: one deliberately too short (simulates a slow/overloaded server), one realistic (what you'd actually use in production).

In [ ]:
PRIMARY_DEPLOYMENT = "gpt-5.4-nano"
FALLBACK_DEPLOYMENT = "gpt-4.1-nano"

SHORT_TIMEOUT = 0.01
NORMAL_TIMEOUT = 20.0

## Cell 5 — Normal call: the happy path


In [ ]:
prompt = "What is the difference between latency and throughput?"

response = client.responses.create(
    model=PRIMARY_DEPLOYMENT,
    input=prompt,
    timeout=NORMAL_TIMEOUT
)

print(response.output_text)

**Latency** and **throughput** describe different performance aspects of a system:

### Latency
- **What it is:** The **time it takes** to complete a single operation (e.g., one request).
- **Measured as:** milliseconds (ms), microseconds (µs), etc.
- **Example:** “How long does it take for one web request to return a response?”

### Throughput
- **What it is:** The **amount of work** the system can handle over time.
- **Measured as:** requests per second (RPS), transactions per second (TPS), bytes/sec, etc.
- **Example:** “How many web requests per second can the server handle?”

### Quick intuition
- **Latency** = *how fast one item finishes*
- **Throughput** = *how many items finish per second*

### Relationship (common trade-off)
Systems often optimize one at the expense of the other:
- Increasing throughput (more parallelism) can sometimes **increase latency** due to contention.
- Reducing latency (e.g., strict timeouts, smaller batches) can sometimes **reduce throughput**.

If yo

## Break it: raw crash, no handling


In [ ]:
response = client.responses.create(
    model=PRIMARY_DEPLOYMENT,
    input=prompt,
    timeout=SHORT_TIMEOUT
)

print(response.output_text)

APITimeoutError: Request timed out.

## Add retry only


In [ ]:
@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_random_exponential(min=1, max=10),
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, APITimeoutError))
)
def call_with_retry(model: str, user_message: str, timeout: float):
    print(f"[{time.strftime('%H:%M:%S')}] Attempting call to {model} (timeout={timeout}s)...")
    response = client.responses.create(
        model=model,
        input=user_message,
        timeout=timeout
    )

    return response.output_text

## Call it with the short timeout



In [ ]:
call_with_retry(PRIMARY_DEPLOYMENT, prompt, timeout=SHORT_TIMEOUT)

[17:30:55] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:30:58] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:31:00] Attempting call to gpt-5.4-nano (timeout=0.01s)...


APITimeoutError: Request timed out.

## Add fallback on top


In [ ]:
try:
    result = call_with_retry(PRIMARY_DEPLOYMENT, prompt, timeout=SHORT_TIMEOUT)
except Exception as primary_error:
    print(f"\n[WARNING] Primary failed after retries: {primary_error}")
    print("[FAILOVER] Switching to fallback deployment...")
    result = call_with_retry(FALLBACK_DEPLOYMENT, prompt, timeout=NORMAL_TIMEOUT)

print("\n**************FINAL RESULT:*******************\n\n", result)

[17:31:05] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:31:07] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:31:10] Attempting call to gpt-5.4-nano (timeout=0.01s)...

[WARNING] Primary failed after retries: Request timed out.
[FAILOVER] Switching to fallback deployment...
[17:31:12] Attempting call to gpt-4.1-nano (timeout=20.0s)...

**************FINAL RESULT:*******************

 **Latency** and **throughput** are two key metrics used to evaluate the performance of a network or system, but they measure different aspects:

### 1. Latency
- **Definition:** The time it takes for a single unit of data (like a packet) to travel from the sender to the receiver.
- **Measurement:** Usually measured in **milliseconds (ms)**.
- **Implication:** Lower latency means quicker response times, which is critical for applications like online gaming, video conferencing, and real-time communication.
- **Example:** If you click a link and the webpage loads in 200ms, that 200ms is the 

## Put it together: one reusable function

In [ ]:
def resilient_chat_responses(user_message: str) -> str:
    try:
        return call_with_retry(PRIMARY_DEPLOYMENT, user_message, timeout=SHORT_TIMEOUT)
    except Exception:
        print("[FAILOVER] Primary unavailable — switching to fallback deployment...")
        return call_with_retry(FALLBACK_DEPLOYMENT, user_message, timeout=NORMAL_TIMEOUT)

## Test it with one new prompt


In [ ]:
output = resilient_chat_responses("What is the CAP theorem in computer science?")

print("\n**************FINAL RESULT:*******************\n\n", output)

[17:33:22] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:33:24] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[17:33:26] Attempting call to gpt-5.4-nano (timeout=0.01s)...
[FAILOVER] Primary unavailable — switching to fallback deployment...
[17:33:28] Attempting call to gpt-4.1-nano (timeout=20.0s)...

**************FINAL RESULT:*******************

 The CAP theorem, also known as Brewer's theorem, is a fundamental principle in distributed systems that states it's impossible for a distributed data store to simultaneously provide all three of the following guarantees:

1. **Consistency (C):** All nodes see the same data at the same time. That is, after a write completes, any subsequent reads will return the most recent write or an error.

2. **Availability (A):** Every request receives a (non-error) response, without the guarantee that it contains the most recent data. The system remains operational and responsive.

3. **Partition Tolerance (P):** The system continues to o